In [13]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Flatten,
    Lambda
)

# Parameters
embedding_dim = 64
input_shape = (28, 28)

# Embedding Network
def create_embedding_network():

    inputs = Input(shape=input_shape)

    x = Flatten()(inputs)

    x = Dense(
        128,
        activation='relu'
    )(x)

    x = Dense(
        64,
        activation='relu'
    )(x)

    outputs = Dense(embedding_dim)(x)

    return Model(inputs, outputs)

# Create Embedding Network
embedding_network = create_embedding_network()

# Inputs
anchor_input = Input(shape=input_shape)
positive_input = Input(shape=input_shape)
negative_input = Input(shape=input_shape)

# Generate Embeddings
anchor_embedding = embedding_network(anchor_input)
positive_embedding = embedding_network(positive_input)
negative_embedding = embedding_network(negative_input)

# Distance Function
def euclidean_distance(vectors):

    x, y = vectors

    return tf.reduce_sum(
        tf.square(x - y),
        axis=1,
        keepdims=True
    )

# Distances
positive_distance = Lambda(
    euclidean_distance
)([anchor_embedding, positive_embedding])

negative_distance = Lambda(
    euclidean_distance
)([anchor_embedding, negative_embedding])

# Difference
output = Lambda(
    lambda tensors: tensors[0] - tensors[1]
)([positive_distance, negative_distance])

# Build Model
triplet_model = Model(
    inputs=[
        anchor_input,
        positive_input,
        negative_input
    ],
    outputs=output
)

# Triplet Loss
def triplet_loss(y_true, y_pred):

    margin = 1.0

    return tf.reduce_mean(
        tf.maximum(y_pred + margin, 0.0)
    )

# Compile Model
triplet_model.compile(
    optimizer='adam',
    loss=triplet_loss
)

# Generate Dummy Data
num_samples = 100

anchor_data = np.random.random(
    (num_samples, 28, 28)
)

positive_data = np.random.random(
    (num_samples, 28, 28)
)

negative_data = np.random.random(
    (num_samples, 28, 28)
)

dummy_labels = np.zeros((num_samples, 1))

# Train Model
history = triplet_model.fit(
    [
        anchor_data,
        positive_data,
        negative_data
    ],
    dummy_labels,
    epochs=10,
    batch_size=16
)

print("\nTriplet Loss Model Trained Successfully")

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 152ms/step - loss: 1.3839
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2981 
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0757 
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0112     
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0094     
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0000e+00 
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0000e+00 
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0000e+00 
Epoch 9/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0000e+00 
Epoch 10/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0000e+00 

Triplet Loss Model Trained Successfully
